# Genshin Wiki JSONL 数据探查

从本地 `genshin.jsonl` 读取前 100 条记录，并查看单条文档的字段与嵌套结构。

In [ ]:
import json
from pathlib import Path
from pprint import pprint

JSONL_PATH = Path("/Users/binzhou/Downloads/genshin.jsonl")
N = 100


def load_first_n_jsonl(path: Path, n: int) -> list[dict]:
    """逐行读取 JSONL，解析为 dict，最多取前 n 条。"""
    records: list[dict] = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
            if len(records) >= n:
                break
    return records


def describe_structure(obj, name: str = "value", depth: int = 0, max_depth: int = 6) -> None:
    """递归打印 Python 对象的结构（类型、键、列表长度与元素类型）。"""
    indent = "  " * depth
    if depth > max_depth:
        print(f"{indent}{name}: ... (max depth)")
        return
    if obj is None:
        print(f"{indent}{name}: NoneType")
    elif isinstance(obj, bool):
        print(f"{indent}{name}: bool")
    elif isinstance(obj, int):
        print(f"{indent}{name}: int")
    elif isinstance(obj, float):
        print(f"{indent}{name}: float")
    elif isinstance(obj, str):
        preview = obj[:80] + "..." if len(obj) > 80 else obj
        preview = preview.replace("\n", "\\n")
        print(f"{indent}{name}: str (len={len(obj)}) e.g. {preview!r}")
    elif isinstance(obj, list):
        print(f"{indent}{name}: list (len={len(obj)})")
        if not obj:
            print(f"{indent}  (empty)")
        else:
            print(f"{indent}  item type: {type(obj[0]).__name__}")
            describe_structure(obj[0], "[0]", depth + 1, max_depth)
    elif isinstance(obj, dict):
        print(f"{indent}{name}: dict (keys={list(obj.keys())})")
        for k, v in obj.items():
            describe_structure(v, str(k), depth + 1, max_depth)
    else:
        print(f"{indent}{name}: {type(obj).__name__}")


records = load_first_n_jsonl(JSONL_PATH, N)
print(f"文件: {JSONL_PATH}")
print(f"已加载条数: {len(records)} (请求前 {N} 条)\n")

if not records:
    print("无数据，请确认文件路径与编码。")
else:
    print("=== 单条文档结构（以第 1 条为例，递归）===\n")
    describe_structure(records[0], "record")

    print("\n=== 第 1 条文档键概览 ===")
    pprint({k: type(v).__name__ for k, v in records[0].items()})

    print("\n=== 第 1 条 url（截断显示）===")
    print(records[0].get("url", "")[:120], "..." if len(records[0].get("url", "")) > 120 else "")

    print("\n=== sections[0] 示例（若存在）===")
    secs = records[0].get("sections")
    if isinstance(secs, list) and secs:
        pprint(secs[0], width=120, depth=2)

文件: /Users/binzhou/Downloads/genshin.jsonl
已加载条数: 100 (请求前 100 条)

=== 单条文档结构（以第 1 条为例，递归）===

record: dict (keys=['url', 'sections', 'category', 'markdown'])
  url: str (len=69) e.g. 'https://genshin-impact.fandom.com/wiki/%22Black_Nacre%22_Promo_Poster'
  sections: list (len=8)
    item type: dict
    [0]: dict (keys=['title', 'level', 'markdown'])
      title: str (len=26) e.g. '"Black Nacre" Promo Poster'
      level: int
      markdown: str (len=28) e.g. '# "Black Nacre" Promo Poster'
  category: list (len=10)
    item type: str
    [0]: str (len=11) e.g. 'Furnishings'
  markdown: str (len=1150) e.g. '# "Black Nacre" Promo Poster\\n\\nOne copy of "Black Nacre" Promo Poster can be obta...'

=== 第 1 条文档键概览 ===
{'category': 'list', 'markdown': 'str', 'sections': 'list', 'url': 'str'}

=== 第 1 条 url（截断显示）===
https://genshin-impact.fandom.com/wiki/%22Black_Nacre%22_Promo_Poster 

=== sections[0] 示例（若存在）===
{'level': 0, 'markdown': '# "Black Nacre" Promo Poster', 'title': '"Black Nacre"